# <font color="#418FDE" size="6.5" uppercase>**Production Integration**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Design production-ready pipelines with consistent preprocessing and leakage prevention. 
- Use output configuration, feature names, and array handling to integrate with downstream systems. 
- Plan and evaluate a capstone scikit-learn project from data framing to validation. 


## **1. Reliable Pipelines**

### **1.1. Consistent Preprocessing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_01_01.jpg?v=1784045267" width="250">



>* Transform new data like training data
>* Keep preprocessing inside the production pipeline

>* Keep feature meanings consistent everywhere
>* Bundle preprocessing with the model

>* Pipelines make preprocessing understandable and maintainable
>* Versioned changes keep deployments reliable



In [ ]:
#@title Python Code - Consistent Preprocessing

# This example builds one reliable preprocessing pipeline.
# Training data alone learns every preprocessing rule.
# New rows receive the same transformations automatically.

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Small raw data includes missing and categorical values.
data = pd.DataFrame(
    {
        "age": [22, 35, 58, 45, 29, 63, 41, 37, 52, 48, 31, 60],
        "balance": [1200, 3400, np.nan, 2800, 900, 5200, 3100, 2600, 4800, np.nan, 1500, 6100],
        "plan": ["basic", "plus", "premium", "plus", "basic", "premium", "plus", "basic", "premium", "plus", "basic", "premium"],
        "churned": [0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1],
    }
)

# Separate raw features from the target label.
features = data[["age", "balance", "plan"]]
target = data["churned"]

# Stratification keeps both classes represented in each split.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.33, random_state=42, stratify=target
)

# Numeric columns are imputed and scaled inside the pipeline.
numeric_steps = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Categorical columns are encoded with unknown categories handled safely.
categorical_steps = Pipeline(
    [("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
)

# ColumnTransformer applies the right preprocessing to each column.
preprocessor = ColumnTransformer(
    [("num", numeric_steps, ["age", "balance"]), ("cat", categorical_steps, ["plan"])]
)

# The model and preprocessing are fitted as one unit.
model_pipeline = Pipeline(
    [("preprocess", preprocessor), ("model", LogisticRegression(random_state=42))]
)

# Fitting here learns medians, scaling, encodings, and model weights.
model_pipeline.fit(X_train, y_train)

# The test set is transformed using only training-learned rules.
predictions = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# A new production row can contain missing and unseen values.
new_customer = pd.DataFrame(
    {"age": [50], "balance": [np.nan], "plan": ["enterprise"]}
)

# The same pipeline safely preprocesses and predicts the new row.
new_prediction = model_pipeline.predict(new_customer)[0]
feature_names = model_pipeline.named_steps["preprocess"].get_feature_names_out()

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"Test accuracy with one fitted pipeline: {accuracy:.2f}")
print(f"Transformed feature count: {len(feature_names)}")
print(f"First transformed features: {list(feature_names[:5])}")
print(f"New customer churn prediction: {int(new_prediction)}")



### **1.2. Prevent Data Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_01_02.jpg?v=1784045271" width="250">



>* Avoid using future information during model development
>* Use only features available at prediction time

>* Fit preprocessing only on training data
>* Apply unchanged steps to unseen data

>* Split by time and entity groups
>* Avoid answer proxies using domain context



In [ ]:
#@title Python Code - Prevent Data Leakage

# This example shows leakage from early preprocessing.
# Pipelines keep test information out of training.
# The printed scores compare leaky and safe workflows.

import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Create many noisy features where leakage can look impressive.
features, target = make_classification(
    n_samples=600,
    n_features=200,
    n_informative=5,
    random_state=42,
)

# Validate the generated table before modeling.
if features.shape != (600, 200):
    raise ValueError("Expected 600 rows and 200 features.")

# Split before fitting any preprocessing step.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Leaky workflow fits feature selection on all rows first.
leaky_selector = SelectKBest(score_func=f_classif, k=10)
leaky_features = leaky_selector.fit_transform(features, target)

# The model then sees a test set already used by preprocessing.
leaky_train, leaky_test, leaky_y_train, leaky_y_test = train_test_split(
    leaky_features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Train one simple classifier on the leaky features.
leaky_model = LogisticRegression(max_iter=1000, random_state=42)
leaky_model.fit(leaky_train, leaky_y_train)
leaky_accuracy = accuracy_score(leaky_y_test, leaky_model.predict(leaky_test))

# Safe workflow learns feature selection inside the pipeline.
safe_pipeline = Pipeline(
    steps=[
        ("select", SelectKBest(score_func=f_classif, k=10)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# The pipeline fits preprocessing only on training rows.
safe_pipeline.fit(train_features, train_target)
safe_predictions = safe_pipeline.predict(test_features)
safe_accuracy = accuracy_score(test_target, safe_predictions)

# Print concise results that reveal the leakage effect.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Leaky accuracy: {leaky_accuracy:.3f}")
print(f"Pipeline accuracy: {safe_accuracy:.3f}")
print("Lesson: split first, then fit preprocessing inside the pipeline.")



### **1.3. Controlled Randomness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_01_03.jpg?v=1784045269" width="250">



>* Random steps can change pipeline results
>* Fixed seeds make experiments repeatable

>* Fixed splits make evaluations comparable
>* Repeated validation reveals true performance variability

>* Fixed randomness enables reproducible production models
>* Retrain and monitor as data changes



In [ ]:
#@title Python Code - Controlled Randomness

# This example shows why fixed seeds matter.
# Controlled randomness makes pipeline results repeatable.
# You will compare unstable and stable evaluations.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Build preprocessing and modeling into one leakage-safe pipeline.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)

# Two splits without random_state can evaluate different test records.
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X, y, test_size=0.25, stratify=y
)

pipeline.fit(X_train_a, y_train_a)
accuracy_a = accuracy_score(y_test_a, pipeline.predict(X_test_a))

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y, test_size=0.25, stratify=y
)

pipeline.fit(X_train_b, y_train_b)
accuracy_b = accuracy_score(y_test_b, pipeline.predict(X_test_b))

# Fixed random_state makes the split reproducible across repeated runs.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

pipeline.fit(X_train_c, y_train_c)
accuracy_c = accuracy_score(y_test_c, pipeline.predict(X_test_c))

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

pipeline.fit(X_train_d, y_train_d)
accuracy_d = accuracy_score(y_test_d, pipeline.predict(X_test_d))

# Print concise results for production-style reproducibility.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Uncontrolled split accuracies: {accuracy_a:.3f} and {accuracy_b:.3f}")
print(f"Controlled split accuracies: {accuracy_c:.3f} and {accuracy_d:.3f}")
print(f"Controlled test sets identical: {(X_test_c == X_test_d).all()}")



## **2. Output Integration**

### **2.1. Pandas Output Configuration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_02_01.jpg?v=1784045275" width="250">



>* Preserve column meaning after preprocessing
>* Avoid fragile array position mappings

>* Labeled outputs keep environments consistent
>* Column names reveal downstream mismatches early

>* Balance labeled output with performance needs
>* Choose formats deliberately for maintainable pipelines



In [ ]:
#@title Python Code - Pandas Output Configuration

# Demonstrate pandas output from preprocessing pipelines.
# Preserve feature names after mixed transformations.
# Compare labeled output with plain arrays.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Build a tiny customer table in memory.
customers = pd.DataFrame(
    {
        "tenure_months": [2, 14, 36, 48],
        "monthly_charge": [29.9, 55.0, 72.5, 88.0],
        "contract_type": ["month", "one_year", "month", "two_year"],
    }
)

# Separate numeric and categorical preprocessing steps.
numeric_features = ["tenure_months", "monthly_charge"]
categorical_features = ["contract_type"]

# Combine transformations while keeping column meaning explicit.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(sparse_output=False), categorical_features),
    ]
)

# Fit only once, then inspect the default array output.
array_pipeline = Pipeline(steps=[("preprocess", preprocessor)])
array_output = array_pipeline.fit_transform(customers)

# Configure the same pipeline to return a pandas DataFrame.
pandas_pipeline = Pipeline(steps=[("preprocess", preprocessor)])
pandas_pipeline.set_output(transform="pandas")
pandas_output = pandas_pipeline.fit_transform(customers)

# Validate that labels and rows were preserved.
if pandas_output.shape[0] != customers.shape[0]:
    raise ValueError("Transformed rows should match input rows.")

print("scikit-learn version:", sklearn.__version__)
print("Default output type:", type(array_output).__name__)
print("Pandas output type:", type(pandas_output).__name__)
print("Named transformed columns:", list(pandas_output.columns))
print("First transformed row:")
print(pandas_output.head(1).round(2))



### **2.2. Feature Name Tracking**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_02_02.jpg?v=1784045277" width="250">



>* Track names through preprocessing changes
>* Keep transformed features understandable for integration

>* Verify feature alignment across production environments
>* Use names to improve team communication

>* Stable names enable interpretation and auditing
>* Tracked features support governance and maintenance



In [ ]:
#@title Python Code - Feature Name Tracking

# Track transformed feature names for production integration.
# Pipelines preserve meaning after preprocessing changes columns.
# The output shows aligned names and values.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Small in-memory data keeps the example easy to inspect.
customers = pd.DataFrame(
    {
        "contract_type": ["month", "year", "month", "two_year"],
        "monthly_charges": [70.0, 45.0, 85.0, 60.0],
        "tenure_months": [5, 24, 3, 48],
    }
)

# Separate column lists make the preprocessing contract explicit.
categorical_columns = ["contract_type"]
numeric_columns = ["monthly_charges", "tenure_months"]

# The transformer learns category order and scaling from named columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(sparse_output=False), categorical_columns),
        ("number", StandardScaler(), numeric_columns),
    ]
)

# Pandas output keeps transformed values attached to feature names.
preprocessor.set_output(transform="pandas")
transformed_customers = preprocessor.fit_transform(customers)

# Validate that names and transformed columns stay aligned.
feature_names = list(preprocessor.get_feature_names_out())
if feature_names != list(transformed_customers.columns):
    raise ValueError("Feature names do not match transformed columns.")

# Show the production-friendly contract and one transformed record.
print("scikit-learn version:", sklearn.__version__)
print("Raw input columns:", list(customers.columns))
print("Transformed feature count:", len(feature_names))
print("First five feature names:", feature_names[:5])
print("First transformed row:")
print(transformed_customers.head(1).round(2))



### **2.3. Array API Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_02_03.jpg?v=1784045273" width="250">



>* Arrays need consistent structure across systems
>* Wrong feature order causes silent prediction errors

>* Define exact array structure and meanings
>* Validate dimensions, types, and value ranges

>* Support varied array formats and compute backends
>* Balance sparse efficiency with feature traceability



In [ ]:
#@title Python Code - Array API Handling

# This example protects array feature order.
# Feature names define the production contract.
# Validation catches swapped or missing inputs.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Training data keeps names for safe preprocessing.
training_data = pd.DataFrame(
    {
        "age": [22, 35, 58, 45, 29, 63, 41, 52],
        "balance": [1200, 5400, 2100, 7600, 3300, 1800, 6200, 4100],
        "segment": ["new", "vip", "standard", "vip", "new", "standard", "vip", "new"],
    }
)

# The target is small and deterministic.
target = np.array([0, 1, 0, 1, 0, 0, 1, 0])

# The pipeline records preprocessing and model steps together.
preprocessor = ColumnTransformer(
    [
        ("numeric", StandardScaler(), ["age", "balance"]),
        ("category", OneHotEncoder(handle_unknown="ignore"), ["segment"]),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(random_state=42, max_iter=200)),
    ]
)

# Fitting on named columns creates the expected contract.
model.fit(training_data, target)
expected_columns = list(model.feature_names_in_)
output_names = model.named_steps["preprocess"].get_feature_names_out()

# Production arrays need an explicit column order.
production_array = np.array([[40, 5000, "vip"]], dtype=object)
production_frame = pd.DataFrame(production_array, columns=expected_columns)

# Validate shape before the array reaches the model.
if production_array.shape[1] != len(expected_columns):
    raise ValueError("Production array has the wrong number of columns.")

# A swapped array can pass shape checks but change meaning.
swapped_array = np.array([[5000, 40, "vip"]], dtype=object)
swapped_frame = pd.DataFrame(swapped_array, columns=expected_columns)

correct_probability = model.predict_proba(production_frame)[0, 1]
swapped_probability = model.predict_proba(swapped_frame)[0, 1]
transformed = model.named_steps["preprocess"].transform(production_frame)

print("scikit-learn version:", sklearn.__version__)
print("Expected raw array order:", expected_columns)
print("Transformed feature names:", list(output_names))
print("Transformed array shape:", transformed.shape)
print("Correct order probability:", round(float(correct_probability), 3))
print("Swapped numeric order probability:", round(float(swapped_probability), 3))
print("Lesson: arrays need a documented order and boundary validation.")



## **3. Capstone Planning**

### **3.1. Choosing Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_03_01.jpg?v=1784045261" width="250">



>* Match estimator type to the task
>* Use baselines and meaningful evaluation metrics

>* Match estimators to data characteristics
>* Balance performance with practical constraints

>* Compare candidates with consistent validation.
>* Choose practical, fair, maintainable models.



In [ ]:
#@title Python Code - Choosing Estimators

# Compare estimators using one fair validation plan.
# Pipelines prevent leakage during preprocessing and modeling.
# Results show evidence for choosing a capstone estimator.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Use one honest split for both candidate estimators.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# A baseline defines what improvement means.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

# A pipeline fits scaling only on training data.
logistic_pipeline = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
logistic_pipeline.fit(X_train, y_train)
model_predictions = logistic_pipeline.predict(X_test)

# Compare candidates with the same metric and test data.
baseline_score = balanced_accuracy_score(y_test, baseline_predictions)
model_score = balanced_accuracy_score(y_test, model_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Task: classification with {X.shape[0]} rows and {X.shape[1]} features")
print(f"Baseline balanced accuracy: {baseline_score:.3f}")
print(f"Pipeline logistic balanced accuracy: {model_score:.3f}")
print("Chosen estimator: pipeline logistic regression, if operations allow it")



### **3.2. Resource Planning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_03_02.jpg?v=1784045265" width="250">



>* Estimate resources before modeling begins
>* Match strategy to constraints and scope

>* Plan resources across the full project lifecycle
>* Prioritize experiments and preserve validation time

>* Assign roles and preserve reproducible project records
>* Plan fallbacks and validate results carefully



### **3.3. Capstone Project**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_32/Lecture_B/image_03_03.jpg?v=1784045263" width="250">



>* Frame the problem before modeling
>* Match predictions to real decisions

>* Build reproducible preprocessing and modeling pipelines
>* Prevent leakage with appropriate validation splits

>* Balance metrics with real-world model tradeoffs
>* Validate, monitor, and plan production use



In [ ]:
#@title Python Code - Capstone Project

# Plan a capstone workflow with leakage prevention.
# Pipelines keep preprocessing inside validation folds.
# Results compare baseline and production-ready evaluation.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer(as_frame=True)
features = data.data
target = data.target

# Validate the basic capstone framing assumptions.
if len(features) != len(target):
    raise ValueError("Feature rows and target labels must match.")

# Split before preprocessing to avoid test-data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Build a baseline that ignores feature values.
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

# Build one reproducible pipeline for preprocessing and modeling.
model_pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)
model_pipeline.fit(X_train, y_train)
model_predictions = model_pipeline.predict(X_test)

# Evaluate with a metric that respects both classes.
baseline_score = balanced_accuracy_score(y_test, baseline_predictions)
model_score = balanced_accuracy_score(y_test, model_predictions)

# Show the production planning checklist and validation result.
print("scikit-learn version:", sklearn.__version__)
print("Problem frame: predict diagnosis class before final evaluation.")
print("Leakage control: split first, then fit preprocessing inside Pipeline.")
print("Baseline balanced accuracy:", round(baseline_score, 3))
print("Pipeline balanced accuracy:", round(model_score, 3))
print("Capstone decision: compare against baseline before deployment planning.")



# <font color="#418FDE" size="6.5" uppercase>**Production Integration**</font>


In this lecture, you learned to:
- Design production-ready pipelines with consistent preprocessing and leakage prevention. 
- Use output configuration, feature names, and array handling to integrate with downstream systems. 
- Plan and evaluate a capstone scikit-learn project from data framing to validation. 

<font color='yellow'>Congratulations on completing this course!</font>